<a href="https://colab.research.google.com/github/westiion/5g-throughput-prediction/blob/main/5g-throughput-prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5G Throughput Prediction

# 1. Setup: Dataset 다운로드

!git clone https://github.com/uccmisl/5Gdataset.git
!unzip -q 5Gdataset/5G-production-dataset.zip -d 5Gdataset/
!ls 5Gdataset
!ls 5Gdataset/5G-production-dataset/

In [ ]:
import pandas as pd
import glob
csv_files=glob.glob('5Gdataset/**/*.csv', recursive=True)
print(f"총 CSV 파일 수: {len(csv_files)}")
print("처음 3개:", csv_files[:3])

df=pd.read_csv(csv_files[0])
print("\nShape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nHead:")
df.head()

총 CSV 파일 수: 83
처음 3개: ['5Gdataset/5G-production-dataset/Download/Static/B_2020.02.14_13.21.26.csv', '5Gdataset/5G-production-dataset/Download/Static/B_2020.01.16_10.43.34.csv', '5Gdataset/5G-production-dataset/Download/Static/B_2020.02.13_13.57.29.csv']

Shape: (1013, 26)

Columns: ['Timestamp', 'Longitude', 'Latitude', 'Speed', 'Operatorname', 'CellID', 'NetworkMode', 'RSRP', 'RSRQ', 'SNR', 'CQI', 'RSSI', 'DL_bitrate', 'UL_bitrate', 'State', 'PINGAVG', 'PINGMIN', 'PINGMAX', 'PINGSTDEV', 'PINGLOSS', 'CELLHEX', 'NODEHEX', 'LACHEX', 'RAWCELLID', 'NRxRSRP', 'NRxRSRQ']

Head:


,Timestamp,Longitude,Latitude,Speed,Operatorname,CellID,NetworkMode,RSRP,RSRQ,SNR,...,PINGMIN,PINGMAX,PINGSTDEV,PINGLOSS,CELLHEX,NODEHEX,LACHEX,RAWCELLID,NRxRSRP,NRxRSRQ
0,2020.02.14_13.21.26,-8.394571,51.886244,0,B,11,5G,-103,-15,5.0,...,-,-,-,-,B,A4DF,9CBA,10805003,-105,-10
1,2020.02.14_13.21.27,-8.394571,51.886244,0,B,11,5G,-103,-15,5.0,...,-,-,-,-,B,A4DF,9CBA,10805003,-105,-10
2,2020.02.14_13.21.27,-8.394571,51.886244,0,B,11,5G,-102,-15,1.0,...,-,-,-,-,B,A4DF,9CBA,10805003,-105,-10
3,2020.02.14_13.21.28,-8.394571,51.886244,0,B,11,5G,-102,-15,1.0,...,-,-,-,-,B,A4DF,9CBA,10805003,-105,-10
4,2020.02.14_13.21.29,-8.394571,51.886244,0,B,11,5G,-103,-16,-2.0,...,-,-,-,-,B,A4DF,9CBA,10805003,-103,-11


# 2. EDA: 데이터 탐색

In [ ]:
# 2.1 첫 trace 컬럼·결측치 확인
print("Describe:")
print(df.describe())

print("\nMissing values:")
print(df.isna().sum())

print("\nDtypes:")
print(df.dtypes)

Describe:
          Longitude      Latitude   Speed  CellID         RSRP         RSRQ  \
count  1.013000e+03  1.013000e+03  1013.0  1013.0  1013.000000  1013.000000   
mean  -8.394571e+00  5.188624e+01     0.0    11.0  -102.870681   -14.717670   
std    1.013024e-13  7.180026e-13     0.0     0.0     0.879082     2.025347   
min   -8.394571e+00  5.188624e+01     0.0    11.0  -106.000000   -20.000000   
25%   -8.394571e+00  5.188624e+01     0.0    11.0  -103.000000   -16.000000   
50%   -8.394571e+00  5.188624e+01     0.0    11.0  -103.000000   -15.000000   
75%   -8.394571e+00  5.188624e+01     0.0    11.0  -102.000000   -13.000000   
max   -8.394571e+00  5.188624e+01     0.0    11.0  -101.000000   -10.000000   

               SNR          CQI         RSSI     DL_bitrate   UL_bitrate  \
count  1013.000000  1013.000000  1013.000000    1013.000000  1013.000000   
mean     -1.370188    10.248766  -104.068115  121021.051333   174.313919   
std       3.222049     2.053463     1.405909   885

In [ ]:
# 2.2 시나리오별 trace 분포
import pandas as pd
import glob
from collections import Counter

csv_files = sorted(glob.glob('5Gdataset/**/*.csv', recursive=True))
print(f"Total CSV files: {len(csv_files)}\n")

def parse_scenario(filepath):
    parts = filepath.split('/')
    return parts[2], parts[3]

scenarios = [parse_scenario(f) for f in csv_files]
print("App별:")
for app, c in Counter(s[0] for s in scenarios).most_common():
    print(f"  {app}: {c}")

print("\nMobility별:")
for mob, c in Counter(s[1] for s in scenarios).most_common():
    print(f"  {mob}: {c}")

print("\nApp × Mobility:")
for combo, c in sorted(Counter(scenarios).items()):
    print(f"  {combo[0]} / {combo[1]}: {c}")

Total CSV files: 83

App별:
  Netflix: 33
  Amazon_Prime: 29
  Download: 21

Mobility별:
  Driving: 60
  Static: 23

App × Mobility:
  Amazon_Prime / Driving: 21
  Amazon_Prime / Static: 8
  Download / Driving: 16
  Download / Static: 5
  Netflix / Driving: 23
  Netflix / Static: 10


In [ ]:
# 2.3 Driving trace 변동성 확인
driving_files = [f for f in csv_files if '/Driving/' in f]
print(f"Driving traces: {len(driving_files)}\n")

df_drive = pd.read_csv(driving_files[0])
print(f"File: {driving_files[0]}")
print(f"Shape: {df_drive.shape}\n")

print("NetworkMode value counts:")
print(df_drive['NetworkMode'].value_counts())

print(f"\nCellID nunique (handover 횟수 proxy): {df_drive['CellID'].nunique()}")
print(f"Speed range: {df_drive['Speed'].min()} ~ {df_drive['Speed'].max()}")

print("\nDL_bitrate describe:")
print(df_drive['DL_bitrate'].describe())

# RF KPI에 "-"가 있는지 확인
print("\nRF KPI에서 '-' 등장 횟수:")
for col in ['RSRP', 'RSRQ', 'SNR', 'CQI', 'RSSI', 'NRxRSRP', 'NRxRSRQ']:
    n_dash = (df_drive[col].astype(str) == '-').sum()
    n_total = len(df_drive)
    if n_dash > 0:
        print(f"  {col}: {n_dash}/{n_total} ({n_dash/n_total*100:.1f}%)")
print("(아무것도 안 찍히면 이 trace엔 '-'가 없음)")

Driving traces: 60

File: 5Gdataset/5G-production-dataset/Amazon_Prime/Driving/Season3-TheExpanse/B_2019.12.01_12.11.21.csv
Shape: (673, 26)

NetworkMode value counts:
NetworkMode
5G     429
LTE    244
Name: count, dtype: int64

CellID nunique (handover 횟수 proxy): 4
Speed range: 0 ~ 64

DL_bitrate describe:
count      673.000000
mean       798.526003
std       2668.826965
min          0.000000
25%          0.000000
50%        279.000000
75%        760.000000
max      42741.000000
Name: DL_bitrate, dtype: float64

RF KPI에서 '-' 등장 횟수:
  RSSI: 76/673 (11.3%)
  NRxRSRP: 128/673 (19.0%)
  NRxRSRQ: 128/673 (19.0%)
(아무것도 안 찍히면 이 trace엔 '-'가 없음)


In [ ]:
# 2.4 전체 데이터셋 결측치 패턴
all_dfs = []
for f in csv_files:
    df_tmp = pd.read_csv(f)
    df_tmp['trace_id'] = f.split('/')[-1].replace('.csv', '')
    df_tmp['application'] = f.split('/')[2]
    df_tmp['mobility'] = f.split('/')[3]
    all_dfs.append(df_tmp)

df_all = pd.concat(all_dfs, ignore_index=True)
print(f"전체 row 수: {len(df_all):,}")
print(f"전체 trace 수: {df_all['trace_id'].nunique()}\n")

# "-"를 NaN으로 변환
df_all_clean = df_all.replace('-', pd.NA)

# 모든 RF/throughput 관련 컬럼을 numeric으로 강제 변환 (변환 실패 → NaN)
numeric_cols = ['RSRP', 'RSRQ', 'SNR', 'CQI', 'RSSI', 'DL_bitrate', 'UL_bitrate',
                'NRxRSRP', 'NRxRSRQ', 'PINGAVG', 'PINGMIN', 'PINGMAX', 'PINGSTDEV', 'PINGLOSS',
                'Speed', 'Longitude', 'Latitude']
for col in numeric_cols:
    if col in df_all_clean.columns:
        df_all_clean[col] = pd.to_numeric(df_all_clean[col], errors='coerce')

print("실제 결측치 비율 (전체 데이터셋):")
missing_pct = (df_all_clean.isna().sum() / len(df_all_clean) * 100).sort_values(ascending=False)
print(missing_pct[missing_pct > 0].round(2))

print("\nNetworkMode 전체 분포:")
print(df_all_clean['NetworkMode'].value_counts(dropna=False))

print("\nDL_bitrate 전체 분포 (kbps):")
print(df_all_clean['DL_bitrate'].describe())

전체 row 수: 188,711
전체 trace 수: 83

실제 결측치 비율 (전체 데이터셋):
PINGMIN      98.30
PINGMAX      98.30
PINGSTDEV    98.30
PINGAVG      98.30
PINGLOSS     98.27
RSSI         28.39
NRxRSRQ      19.73
NRxRSRP      18.28
CQI           9.06
SNR           9.06
RSRQ          1.93
NODEHEX       0.09
dtype: float64

NetworkMode 전체 분포:
NetworkMode
5G       136403
LTE       35212
HSPA+     15880
UMTS        492
HSUPA       470
HSDPA       163
GPRS         78
EDGE         13
Name: count, dtype: int64

DL_bitrate 전체 분포 (kbps):
count    188711.000000
mean      10758.245603
std       36746.787199
min           0.000000
25%           0.000000
50%           2.000000
75%        4480.000000
max      532905.000000
Name: DL_bitrate, dtype: float64


# 3. Data Preparation

In [1]:
# 3.1 모든 trace 통합 + 메타데이터 추가
import pandas as pd
import numpy as np
import glob
import os

csv_files = sorted(glob.glob('5Gdataset/**/*.csv', recursive=True))
print(f"Loading {len(csv_files)} traces...")

all_dfs = []
for f in csv_files:
    parts = f.split('/')
    application = parts[2]   # Amazon_Prime / Netflix / Download
    mobility    = parts[3]   # Driving / Static
    trace_id    = os.path.basename(f).replace('.csv', '')

    df_tmp = pd.read_csv(f)
    df_tmp['trace_id']    = trace_id
    df_tmp['application'] = application
    df_tmp['mobility']    = mobility
    all_dfs.append(df_tmp)

df = pd.concat(all_dfs, ignore_index=True)

print(f"\n전체 row 수:   {len(df):,}")
print(f"전체 trace 수: {df['trace_id'].nunique()}")
print(f"컬럼 수:       {df.shape[1]}")
print(f"\n컬럼 목록:\n{df.columns.tolist()}")

Loading 0 traces...


ValueError: No objects to concatenate

In [ ]:
# 3.2 결측 표기 변환 + 불필요 컬럼 제거 + dtype 통일
df = df.replace('-', np.nan)

columns_to_drop = [
    # 98% 결측 → 사용 불가
    'PINGAVG', 'PINGMIN', 'PINGMAX', 'PINGSTDEV', 'PINGLOSS',
    # CellID와 중복 정보
    'CELLHEX', 'NODEHEX', 'LACHEX', 'RAWCELLID',
    # 거의 단일값 ('B'만 있음)
    'Operatorname',
    # 절대 좌표는 generalization에 도움 안 됨 (Speed는 유지)
    'Longitude', 'Latitude',
    # 의미 모호
    'State',
    # 보조 안테나, 결측 많음
    'NRxRSRP', 'NRxRSRQ',
    # target과 무관 (download 시나리오에서 의미 없음)
    'UL_bitrate',
]
df = df.drop(columns=[c for c in columns_to_drop if c in df.columns])

# 3) 숫자 컬럼을 강제로 numeric으로 변환 (변환 실패 시 NaN)
numeric_cols = ['RSRP', 'RSRQ', 'SNR', 'CQI', 'RSSI', 'DL_bitrate', 'Speed', 'CellID']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 4) 결과 확인
print(f"정리 후 shape: {df.shape}")
print(f"\n남은 컬럼:\n{df.columns.tolist()}")
print(f"\ndtype:\n{df.dtypes}")
print(f"\n각 컬럼 결측치:\n{df.isna().sum()}")
print(f"\n전체 미리보기:")
df.head()

정리 후 shape: (188711, 13)

남은 컬럼:
['Timestamp', 'Speed', 'CellID', 'NetworkMode', 'RSRP', 'RSRQ', 'SNR', 'CQI', 'RSSI', 'DL_bitrate', 'trace_id', 'application', 'mobility']

dtype:
Timestamp       object
Speed            int64
CellID           int64
NetworkMode     object
RSRP             int64
RSRQ           float64
SNR            float64
CQI            float64
RSSI           float64
DL_bitrate       int64
trace_id        object
application     object
mobility        object
dtype: object

각 컬럼 결측치:
Timestamp          0
Speed              0
CellID             0
NetworkMode        0
RSRP               0
RSRQ            3634
SNR            17096
CQI            17097
RSSI           53573
DL_bitrate         0
trace_id           0
application        0
mobility           0
dtype: int64

전체 미리보기:


,Timestamp,Speed,CellID,NetworkMode,RSRP,RSRQ,SNR,CQI,RSSI,DL_bitrate,trace_id,application,mobility
0,2019.12.01_12.11.21,0,2,LTE,-92,-11.0,2.0,8.0,-76.0,29,B_2019.12.01_12.11.21,Amazon_Prime,Driving
1,2019.12.01_12.11.21,0,2,LTE,-93,-10.0,2.0,8.0,-76.0,29,B_2019.12.01_12.11.21,Amazon_Prime,Driving
2,2019.12.01_12.11.22,1,2,LTE,-93,-10.0,2.0,9.0,-78.0,29,B_2019.12.01_12.11.21,Amazon_Prime,Driving
3,2019.12.01_12.11.22,1,2,LTE,-93,-10.0,2.0,9.0,-78.0,2,B_2019.12.01_12.11.21,Amazon_Prime,Driving
4,2019.12.01_12.11.23,1,2,LTE,-93,-14.0,-1.0,10.0,-75.0,1,B_2019.12.01_12.11.21,Amazon_Prime,Driving


# 4. Preprocessing:
# 카테고리 정리 + 핸드오버 Indicator + One-hot Encoding

In [ ]:
df = df.sort_values(['trace_id', 'Timestamp']).reset_index(drop=True)

def categorize_network(mode):
    if mode == '5G':
        return '5G'
    elif mode == 'LTE':
        return 'LTE'
    else:
        return '3GorBelow'  # HSPA+, UMTS, HSUPA, HSDPA, GPRS, EDGE 모두 포함

df['NetworkCategory'] = df['NetworkMode'].apply(categorize_network)

print("NetworkCategory 분포:")
print(df['NetworkCategory'].value_counts())
print(f"5G 비율: {(df['NetworkCategory']=='5G').mean()*100:.1f}%")

df['cellid_changed'] = (
    df.groupby('trace_id')['CellID']
      .transform(lambda x: (x != x.shift(1)).astype(int))
)

first_rows = df.groupby('trace_id').head(1).index
df.loc[first_rows, 'cellid_changed'] = 0

print(f"\n전체 handover(셀 변경) 횟수: {df['cellid_changed'].sum():,}")
print(f"평균 trace당 handover: {df['cellid_changed'].sum() / df['trace_id'].nunique():.1f}")

df = pd.get_dummies(
    df,
    columns=['NetworkCategory', 'application', 'mobility'],
    prefix=['net', 'app', 'mob'],
    dtype=int
)

df = df.drop(columns=['NetworkMode'])

print(f"\n최종 shape: {df.shape}")
print(f"\n컬럼 목록:")
for c in df.columns:
    print(f"  {c}")

NetworkCategory 분포:
NetworkCategory
5G           136403
LTE           35212
3GorBelow     17096
Name: count, dtype: int64
5G 비율: 72.3%

전체 handover(셀 변경) 횟수: 2,113
평균 trace당 handover: 25.5

최종 shape: (188711, 19)

컬럼 목록:
  Timestamp
  Speed
  CellID
  RSRP
  RSRQ
  SNR
  CQI
  RSSI
  DL_bitrate
  trace_id
  cellid_changed
  net_3GorBelow
  net_5G
  net_LTE
  app_Amazon_Prime
  app_Download
  app_Netflix
  mob_Driving
  mob_Static


# 5. Missing Value Handling

In [ ]:
rf_cols_with_missing = ['RSRQ', 'SNR', 'CQI', 'RSSI']

for col in rf_cols_with_missing:
    # 1) Missing indicator 컬럼 추가 (반드시 fill 하기 전에!)
    df[f'{col}_was_missing'] = df[col].isna().astype(int)

    # 2) 같은 trace 안에서 forward-fill
    df[col] = df.groupby('trace_id')[col].transform(lambda x: x.ffill())

    # 3) trace 시작 부분에 남은 NaN(앞에 채울 값이 없는 경우)은 backward-fill
    df[col] = df.groupby('trace_id')[col].transform(lambda x: x.bfill())

    # 4) trace 전체가 NaN인 경우(극히 드묾)는 전체 중앙값으로
    df[col] = df[col].fillna(df[col].median())

print("결측치 처리 후:")
print(df.isna().sum())

print("\nMissing indicator 비율 (원래 결측이었던 비율):")
for col in rf_cols_with_missing:
    pct = df[f'{col}_was_missing'].mean() * 100
    print(f"  {col}_was_missing: {pct:.1f}%")

print(f"\nShape: {df.shape}")

결측치 처리 후:
Timestamp           0
Speed               0
CellID              0
RSRP                0
RSRQ                0
SNR                 0
CQI                 0
RSSI                0
DL_bitrate          0
trace_id            0
cellid_changed      0
net_3GorBelow       0
net_5G              0
net_LTE             0
app_Amazon_Prime    0
app_Download        0
app_Netflix         0
mob_Driving         0
mob_Static          0
RSRQ_was_missing    0
SNR_was_missing     0
CQI_was_missing     0
RSSI_was_missing    0
dtype: int64

Missing indicator 비율 (원래 결측이었던 비율):
  RSRQ_was_missing: 1.9%
  SNR_was_missing: 9.1%
  CQI_was_missing: 9.1%
  RSSI_was_missing: 28.4%

Shape: (188711, 23)


# 6. Feature Engineering: Lag + Rolling Features

In [ ]:
# 다시 한번 정렬 확인 (필수!)
df = df.sort_values(['trace_id', 'Timestamp']).reset_index(drop=True)

# --- (1) Lag features: DL_bitrate의 과거 값 ---
# 1, 2, 3, 5초 전의 throughput
for lag in [1, 2, 3, 5]:
    df[f'DL_bitrate_lag_{lag}'] = (
        df.groupby('trace_id')['DL_bitrate'].shift(lag)
    )

# --- (2) Rolling features: DL_bitrate의 과거 통계 ---
# rolling은 현재까지의 N초 윈도우 (현재값 포함)
for window in [5, 10]:
    df[f'DL_bitrate_rollmean_{window}'] = (
        df.groupby('trace_id')['DL_bitrate']
          .transform(lambda x: x.rolling(window, min_periods=1).mean())
    )
    df[f'DL_bitrate_rollstd_{window}'] = (
        df.groupby('trace_id')['DL_bitrate']
          .transform(lambda x: x.rolling(window, min_periods=2).std())
    )

# --- (3) Rolling mean for key RF KPIs ---
# RSRP, SNR, CQI의 5초 평균 (RF의 단기 추세)
for col in ['RSRP', 'SNR', 'CQI']:
    df[f'{col}_rollmean_5'] = (
        df.groupby('trace_id')[col]
          .transform(lambda x: x.rolling(5, min_periods=1).mean())
    )

# --- (4) 결과 확인 ---
print(f"Lag/rolling 추가 후 shape: {df.shape}")

new_cols = [c for c in df.columns if ('lag' in c) or ('rollmean' in c) or ('rollstd' in c)]
print(f"\n신규 feature ({len(new_cols)}개):")
for c in new_cols:
    print(f"  {c}")

print(f"\n각 신규 컬럼의 NaN 개수 (lag는 trace 시작에서 자동으로 NaN 생김):")
print(df[new_cols].isna().sum())

Lag/rolling 추가 후 shape: (188711, 34)

신규 feature (11개):
  DL_bitrate_lag_1
  DL_bitrate_lag_2
  DL_bitrate_lag_3
  DL_bitrate_lag_5
  DL_bitrate_rollmean_5
  DL_bitrate_rollstd_5
  DL_bitrate_rollmean_10
  DL_bitrate_rollstd_10
  RSRP_rollmean_5
  SNR_rollmean_5
  CQI_rollmean_5

각 신규 컬럼의 NaN 개수 (lag는 trace 시작에서 자동으로 NaN 생김):
DL_bitrate_lag_1           83
DL_bitrate_lag_2          166
DL_bitrate_lag_3          249
DL_bitrate_lag_5          415
DL_bitrate_rollmean_5       0
DL_bitrate_rollstd_5       83
DL_bitrate_rollmean_10      0
DL_bitrate_rollstd_10      83
RSRP_rollmean_5             0
SNR_rollmean_5              0
CQI_rollmean_5              0
dtype: int64


# 7. Target & Train/Test Split

In [ ]:
# 7.1 Target 생성 (log1p) + NaN row 제거
import numpy as np

df['DL_bitrate_next'] = df.groupby('trace_id')['DL_bitrate'].shift(-1)

df['y_log'] = np.log1p(df['DL_bitrate_next'])

print("Target (log1p) 분포:")
print(df['y_log'].describe())
print(f"\nNaN 개수 (각 trace 마지막 row): {df['y_log'].isna().sum()}")

n_before = len(df)
df_clean = df.dropna(subset=['y_log', 'DL_bitrate_lag_5', 'DL_bitrate_rollstd_10']).reset_index(drop=True)
n_after = len(df_clean)

print(f"\nNaN row 제거 전: {n_before:,}")
print(f"NaN row 제거 후: {n_after:,}")
print(f"제거: {n_before - n_after:,} ({(n_before-n_after)/n_before*100:.2f}%)")

Target (log1p) 분포:
count    188628.000000
mean          4.006090
std           4.279614
min           0.000000
25%           0.000000
50%           1.098612
75%           8.409385
max          13.186100
Name: y_log, dtype: float64

NaN 개수 (각 trace 마지막 row): 83

NaN row 제거 전: 188,711
NaN row 제거 후: 188,213
제거: 498 (0.26%)


In [ ]:
# 7.2 Trace-wise Stratified Train/Test Split
from sklearn.model_selection import train_test_split

# 1) 각 trace의 시나리오 정보 추출 (stratify 키 만들기)
scenarios_per_trace = df_clean.groupby('trace_id').agg({
    'app_Amazon_Prime': 'first',
    'app_Netflix':      'first',
    'app_Download':     'first',
    'mob_Driving':      'first',
}).reset_index()

def scenario_label(row):
    if   row['app_Amazon_Prime']: app = 'Amazon'
    elif row['app_Netflix']:      app = 'Netflix'
    else:                          app = 'Download'
    mob = 'Driving' if row['mob_Driving'] else 'Static'
    return f'{app}_{mob}'

scenarios_per_trace['scenario'] = scenarios_per_trace.apply(scenario_label, axis=1)

print("시나리오별 trace 수:")
print(scenarios_per_trace['scenario'].value_counts())

# 2) 80/20 trace-wise stratified split
train_ids, test_ids = train_test_split(
    scenarios_per_trace['trace_id'].values,
    test_size=0.2,
    random_state=42,
    stratify=scenarios_per_trace['scenario'].values
)

# 3) df_clean에서 trace_id로 분할
df_train = df_clean[df_clean['trace_id'].isin(train_ids)].reset_index(drop=True)
df_test  = df_clean[df_clean['trace_id'].isin(test_ids)].reset_index(drop=True)

print(f"\nTrain trace 수: {len(train_ids)}")
print(f"Test trace 수:  {len(test_ids)}")
print(f"Train row 수:   {len(df_train):,}")
print(f"Test row 수:    {len(df_test):,}")

# train/test에 시나리오가 잘 분포됐는지 확인
print("\nTrain 시나리오 분포:")
print(df_train.groupby(['app_Amazon_Prime','app_Netflix','app_Download','mob_Driving']).size())

시나리오별 trace 수:
scenario
Netflix_Driving     23
Amazon_Driving      21
Download_Driving    16
Netflix_Static      10
Amazon_Static        8
Download_Static      5
Name: count, dtype: int64

Train trace 수: 66
Test trace 수:  17
Train row 수:   144,470
Test row 수:    43,743

Train 시나리오 분포:
app_Amazon_Prime  app_Netflix  app_Download  mob_Driving
0                 0            1             0               9602
                                             1              23344
                  1            0             0              28044
                                             1              29240
1                 0            0             0              22618
                                             1              31622
dtype: int64


In [ ]:
# 7.3 X / y 분리 + Feature 목록 확정
# 모델 입력에서 제외할 컬럼들
# - Timestamp, trace_id: 식별자, feature 아님
# - CellID: 임의의 ID라 직접 feature로 부적합 (cellid_changed로 이미 변환)
# - DL_bitrate_next, y_log: target이지 feature 아님
# - DL_bitrate는 KEEP! (시점 t의 현재값 = 가장 강력한 feature)
exclude_cols = ['Timestamp', 'CellID', 'trace_id', 'DL_bitrate_next', 'y_log']

feature_cols = [c for c in df_train.columns if c not in exclude_cols]

print(f"Feature 수: {len(feature_cols)}")
print(f"\nFeature 목록:")
for c in feature_cols:
    print(f"  {c}")

X_train = df_train[feature_cols].values
y_train = df_train['y_log'].values

X_test  = df_test[feature_cols].values
y_test  = df_test['y_log'].values

# 원본 단위(kbps)도 별도 저장 (나중에 metric 계산 시 expm1 검증용)
y_train_actual = df_train['DL_bitrate_next'].values
y_test_actual  = df_test['DL_bitrate_next'].values

print(f"\nX_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train range (log): {y_train.min():.2f} ~ {y_train.max():.2f}")
print(f"y_test range  (log): {y_test.min():.2f} ~ {y_test.max():.2f}")

Feature 수: 31

Feature 목록:
  Speed
  RSRP
  RSRQ
  SNR
  CQI
  RSSI
  DL_bitrate
  cellid_changed
  net_3GorBelow
  net_5G
  net_LTE
  app_Amazon_Prime
  app_Download
  app_Netflix
  mob_Driving
  mob_Static
  RSRQ_was_missing
  SNR_was_missing
  CQI_was_missing
  RSSI_was_missing
  DL_bitrate_lag_1
  DL_bitrate_lag_2
  DL_bitrate_lag_3
  DL_bitrate_lag_5
  DL_bitrate_rollmean_5
  DL_bitrate_rollstd_5
  DL_bitrate_rollmean_10
  DL_bitrate_rollstd_10
  RSRP_rollmean_5
  SNR_rollmean_5
  CQI_rollmean_5

X_train shape: (144470, 31)
X_test shape:  (43743, 31)
y_train range (log): 0.00 ~ 13.19
y_test range  (log): 0.00 ~ 13.09


# 8. Baseline Models

In [ ]:
# 8.1 평가 함수 정의
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def evaluate(y_true_log, y_pred_log, name="Model"):
    """y_log 공간의 true/pred로 양쪽 단위 metric 계산."""
    # 원래 단위(kbps)로 역변환
    y_true_kbps = np.expm1(y_true_log)
    y_pred_kbps = np.clip(np.expm1(y_pred_log), 0, None)  # 음수 throughput 방지

    # kbps 단위 metric
    rmse_kbps = np.sqrt(mean_squared_error(y_true_kbps, y_pred_kbps))
    mae_kbps  = mean_absolute_error(y_true_kbps, y_pred_kbps)
    r2_kbps   = r2_score(y_true_kbps, y_pred_kbps)

    # log 단위 metric
    rmse_log = np.sqrt(mean_squared_error(y_true_log, y_pred_log))
    r2_log   = r2_score(y_true_log, y_pred_log)

    print(f"=== {name} ===")
    print(f"  RMSE (kbps): {rmse_kbps:>12,.1f}")
    print(f"  MAE  (kbps): {mae_kbps:>12,.1f}")
    print(f"  R²   (kbps): {r2_kbps:>12.4f}")
    print(f"  RMSE (log):  {rmse_log:>12.4f}")
    print(f"  R²   (log):  {r2_log:>12.4f}")
    print()

    return {
        'name': name,
        'rmse_kbps': rmse_kbps, 'mae_kbps': mae_kbps, 'r2_kbps': r2_kbps,
        'rmse_log':  rmse_log,  'r2_log':   r2_log,
        'y_pred_log': y_pred_log,
    }

# 결과를 누적할 list (나중에 모델들과 비교용)
results = []

In [ ]:
# 8.2 Baseline 1 — Persistence
# Persistence: 다음 시점 throughput = 현재 시점 throughput
# log 공간에서: y_pred_log = log1p(현재 DL_bitrate)
y_pred_persistence = np.log1p(df_test['DL_bitrate'].values)

result_persistence = evaluate(y_test, y_pred_persistence, name="Persistence baseline")
results.append(result_persistence)

=== Persistence baseline ===
  RMSE (kbps):     20,396.7
  MAE  (kbps):      6,839.4
  R²   (kbps):       0.7879
  RMSE (log):        4.1192
  R²   (log):        0.1584



In [ ]:
# 8.3 Baseline 2 — Linear Regression
from sklearn.linear_model import LinearRegression

# 모델 생성 + 학습
lr = LinearRegression()
lr.fit(X_train, y_train)

# 예측
y_pred_lr = lr.predict(X_test)

result_lr = evaluate(y_test, y_pred_lr, name="Linear Regression")
results.append(result_lr)

# (참고) 모델이 가장 강하게 의존하는 feature 5개 확인
import pandas as pd
coef_df = pd.DataFrame({
    'feature': feature_cols,
    'coef': lr.coef_,
    'abs_coef': np.abs(lr.coef_)
}).sort_values('abs_coef', ascending=False)
print("Linear Regression: 절댓값 큰 계수 top 10")
print(coef_df.head(10).to_string(index=False))

=== Linear Regression ===
  RMSE (kbps):  1,206,527.6
  MAE  (kbps):     48,424.1
  R²   (kbps):    -741.0157
  RMSE (log):        3.1377
  R²   (log):        0.5117

Linear Regression: 절댓값 큰 계수 top 10
         feature      coef  abs_coef
    app_Download  3.505115  3.505115
     app_Netflix -3.227291  3.227291
RSRQ_was_missing -1.776292  1.776292
RSSI_was_missing  0.409568  0.409568
  cellid_changed -0.300957  0.300957
app_Amazon_Prime -0.277824  0.277824
  CQI_rollmean_5 -0.048579  0.048579
         net_LTE -0.047856  0.047856
 CQI_was_missing  0.040603  0.040603
 SNR_was_missing  0.040603  0.040603


In [ ]:
# 8.4 예측 분포 비교 — Log Explosion 진단
# Persistence vs LR 예측 분포 비교
import pandas as pd

pred_persistence_kbps = np.expm1(y_pred_persistence)
pred_lr_kbps = np.clip(np.expm1(y_pred_lr), 0, None)
actual_kbps = np.expm1(y_test)

comparison = pd.DataFrame({
    'Actual':      pd.Series(actual_kbps).describe(),
    'Persistence': pd.Series(pred_persistence_kbps).describe(),
    'Linear Reg':  pd.Series(pred_lr_kbps).describe(),
})
print(comparison.round(0))

         Actual  Persistence   Linear Reg
count   43743.0      43743.0      43743.0
mean    15707.0      15706.0      46964.0
std     44293.0      44293.0    1214143.0
min         0.0          0.0          0.0
25%         0.0          0.0          2.0
50%       102.0        102.0         56.0
75%      9074.0       9073.0        110.0
max    484565.0     484565.0  145813724.0


# 9. Tree-based Models

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import time

# --- Random Forest ---
print("Random Forest 학습 중... (1분 내외)")
t0 = time.time()
rf = RandomForestRegressor(
    n_estimators=100,    # tree 100개 (default)
    max_depth=None,      # 깊이 제한 없음 (default)
    n_jobs=-1,           # 모든 CPU 코어 사용
    random_state=42
)
rf.fit(X_train, y_train)
print(f"학습 시간: {time.time()-t0:.1f}초")

y_pred_rf = rf.predict(X_test)
result_rf = evaluate(y_test, y_pred_rf, name="Random Forest")
results.append(result_rf)

# --- Gradient Boosting ---
print("Gradient Boosting 학습 중... (1~3분)")
t0 = time.time()
gb = GradientBoostingRegressor(
    n_estimators=100,
    max_depth=3,         # GB는 보통 얕은 tree를 많이 쌓음
    learning_rate=0.1,
    random_state=42
)
gb.fit(X_train, y_train)
print(f"학습 시간: {time.time()-t0:.1f}초")

y_pred_gb = gb.predict(X_test)
result_gb = evaluate(y_test, y_pred_gb, name="Gradient Boosting")
results.append(result_gb)

Random Forest 학습 중... (1분 내외)
학습 시간: 271.7초
=== Random Forest ===
  RMSE (kbps):     24,174.7
  MAE  (kbps):      7,457.3
  R²   (kbps):       0.7021
  RMSE (log):        2.2547
  R²   (log):        0.7479

Gradient Boosting 학습 중... (1~3분)
학습 시간: 70.9초
=== Gradient Boosting ===
  RMSE (kbps):     26,100.3
  MAE  (kbps):      8,693.2
  R²   (kbps):       0.6528
  RMSE (log):        2.3770
  R²   (log):        0.7198



In [ ]:
import pandas as pd

results_df = pd.DataFrame([
    {k: v for k, v in r.items() if k != 'y_pred_log'}
    for r in results
])
print(results_df.to_string(index=False))

                name    rmse_kbps     mae_kbps     r2_kbps  rmse_log   r2_log
Persistence baseline 2.039675e+04  6839.445283    0.787939  4.119178 0.158447
   Linear Regression 1.206528e+06 48424.096549 -741.015744  3.137682 0.511710
       Random Forest 2.417470e+04  7457.271827    0.702107  2.254659 0.747871
   Gradient Boosting 2.610033e+04  8693.166291    0.652759  2.377046 0.719757
